In [ ]:
# Connect to source data
!gdown --id 1f4KOh2g406LXdTNC6iNSxSbjXbueKatS
!unzip -q super-ai-engineer-season-6-individual-hackathon-house-recognition.zip

In [ ]:
import pandas as pd
import os

TRAIN_IMG_DIR = "/content/train/train" # ปรับให้ตรงกับที่เก็บภาพ
TEST_IMG_DIR = "/content/test/test"

train_df = pd.read_csv('train.csv')

# ตรวจสอบจำนวนคลาส
num_classes = train_df['class'].nunique()
print(f"Number of classes: {num_classes}")
print(train_df['class'].value_counts())

In [ ]:
import numpy as np
import tensorflow as tf
from sklearn.utils.class_weight import compute_class_weight
from tensorflow.keras.preprocessing.image import ImageDataGenerator

# 1. เตรียม Data และ Class Weights
train_df['class'] = train_df['class'].astype(str)
num_classes = train_df['class'].nunique()

class_weights_array = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(train_df['class']),
    y=train_df['class']
)
class_weights_dict = {i: weight for i, weight in enumerate(class_weights_array)}
print("Class Weights:", class_weights_dict)

# 2. สร้าง ImageDataGenerator
train_datagen = ImageDataGenerator(
    rotation_range=20,
    horizontal_flip=True,
    validation_split=0.2
)

# 3. กำหนด Generator
train_generator = train_datagen.flow_from_dataframe(
    dataframe=train_df,
    directory=TRAIN_IMG_DIR,
    x_col="image_name",
    y_col="class",
    target_size=(384, 384),
    batch_size=16,
    class_mode="categorical",
    subset="training",
    seed=42
)

# กำหนด Generator สำหรับ Validation
val_generator = train_datagen.flow_from_dataframe(
    dataframe=train_df,
    directory=TRAIN_IMG_DIR,
    x_col="image_name",
    y_col="class",
    target_size=(384, 384),
    batch_size=16,
    class_mode="categorical",
    subset="validation",
    seed=42
)

# 4. สร้าง Model (EfficientNetV2B0 + Unfreeze 20 layers ท้าย)
base_model = tf.keras.applications.EfficientNetV2B0(
    input_shape=(384, 384, 3),
    include_top=False,
    weights='imagenet'
)
# ยอมให้ 20 เลเยอร์สุดท้ายเรียนรู้ใหม่ได้
base_model.trainable = True
for layer in base_model.layers[:-20]:
    layer.trainable = False

model = tf.keras.Sequential([
    base_model,
    tf.keras.layers.GlobalAveragePooling2D(),
    tf.keras.layers.Dense(256, activation='relu'),
    tf.keras.layers.Dropout(0.4),
    tf.keras.layers.Dense(num_classes, activation='softmax')
])

# 5. Compile Model
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

# 6. เริ่มเทรน
print("Starting Training...")
history = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=10,
    class_weight=class_weights_dict
)

In [ ]:
import pandas as pd
import numpy as np
import os
import tensorflow as tf
from tensorflow.keras.preprocessing import image

# 1. เช็ก Mapping คลาส (ต้องรัน cell ที่มี train_generator ก่อนนะ)
label_map = train_generator.class_indices
label_map = dict((v, k) for k, v in label_map.items())

# 2. ตั้งค่าโฟลเดอร์ Test
sub_df = pd.read_csv('sample_submission.csv')
predictions = []

print(f"\nStarting Inference with TTA on {len(sub_df)} images...")
not_found_count = 0

for img_id in sub_df['id']:
    img_path = os.path.join(TEST_IMG_DIR, f"{img_id}.jpg")

    if os.path.exists(img_path):
        img = image.load_img(img_path, target_size=(384, 384))

        # ⚠️ เอาการหาร 255 ออก ให้เหลือแค่นี้:
        img_array = image.img_to_array(img)
        img_array = np.expand_dims(img_array, axis=0)

        # ทายผลจากภาพต้นฉบับ
        pred_original = model.predict(img_array, verbose=0)

        # พลิกภาพ (ซ้าย-ขวา) แล้วทายผล
        img_flipped = tf.image.flip_left_right(img_array)
        pred_flipped = model.predict(img_flipped, verbose=0)

        # เอาความมั่นใจมาเฉลี่ยกัน
        pred_final = (pred_original + pred_flipped) / 2.0

        # ดึงคลาสที่โมเดลมั่นใจที่สุดหลังจากเฉลี่ยแล้ว
        pred_idx = np.argmax(pred_final, axis=1)[0]
        pred_class = label_map[pred_idx]
        predictions.append(pred_class)
        # ==========================================

    else:
        not_found_count += 1
        predictions.append('0')

print(f"❌ หาไฟล์ไม่เจอ: {not_found_count} ภาพ")

# 3. บันทึกผลลัพธ์
sub_df['answer'] = predictions
sub_df.to_csv("submission_tta_2.csv", index=False)
print("✅ บันทึกไฟล์ 'submission_tta_2.csv' เรียบร้อยแล้ว ลุยส่งเลย!")

In [ ]:
import pandas as pd
import os

# ⚠️ เช็ค Path ให้ตรงกับ Colab ของคุณ
TRAIN_IMG_DIR = "/content/train/trian"
TEST_IMG_DIR = "/content/test/test"

# 1. เตรียม Train DataFrame
train_df = pd.read_csv('train.csv')
train_df['filepath'] = train_df['image_name'].apply(lambda x: os.path.join(TRAIN_IMG_DIR, x))
train_df['class'] = train_df['class'].astype(str)

# 2. เตรียม Pseudo-Label DataFrame (ใช้ไฟล์ที่ได้ 0.96)
pseudo_df = pd.read_csv('submission_tta_2.csv') # ใช้ไฟล์ตัวเทพสุดของคุณ
pseudo_df['filepath'] = pseudo_df['id'].apply(lambda x: os.path.join(TEST_IMG_DIR, f"{x}.jpg"))
pseudo_df['class'] = pseudo_df['answer'].astype(str)

# 3. ตัดคอลัมน์ให้เหลือแค่ filepath และ class
train_df = train_df[['filepath', 'class']]
pseudo_df = pseudo_df[['filepath', 'class']]

# 4. รวมร่างเข้าด้วยกัน!
combined_df = pd.concat([train_df, pseudo_df], axis=0, ignore_index=True)
print(f"✅ ข้อมูล Train เดิม: {len(train_df)} รูป")
print(f"✅ ข้อมูล Test ที่เติมเข้ามา: {len(pseudo_df)} รูป")
print(f"🚀 ข้อมูลทั้งหมดเตรียมเทรน: {len(combined_df)} รูป")

In [ ]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator
import numpy as np
from sklearn.utils.class_weight import compute_class_weight

# คำนวณ Class Weights ใหม่สำหรับข้อมูลที่รวมร่างแล้ว
class_weights_array = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(combined_df['class']),
    y=combined_df['class']
)
class_weights_dict = {i: weight for i, weight in enumerate(class_weights_array)}

# สร้าง Data Generator (อย่าลืมว่าไม่มี rescale=1./255 แล้วนะ)
train_datagen = ImageDataGenerator(
    rotation_range=20,
    horizontal_flip=True,
    validation_split=0.2
)

# ⚠️ สังเกต directory=None เพราะเรามี filepath เต็มแล้ว
train_generator = train_datagen.flow_from_dataframe(
    dataframe=combined_df,
    directory=None, # <--- ต้องเป็น None
    x_col="filepath",
    y_col="class",
    target_size=(384, 384),
    batch_size=16,
    class_mode="categorical",
    subset="training",
    seed=42
)

val_generator = train_datagen.flow_from_dataframe(
    dataframe=combined_df,
    directory=None, # <--- ต้องเป็น None
    x_col="filepath",
    y_col="class",
    target_size=(384, 384),
    batch_size=16,
    class_mode="categorical",
    subset="validation",
    seed=42
)

In [ ]:
import tensorflow as tf

base_model = tf.keras.applications.EfficientNetV2B0( # ลองใช้ B0 ก่อน ถ้าเวลาเหลือค่อยขยับไป S/M
    input_shape=(384, 384, 3),
    include_top=False,
    weights='imagenet'
)

# แช่แข็งเหมือนเดิม ปล่อยแค่ 20 ชั้นสุดท้าย
base_model.trainable = True
for layer in base_model.layers[:-20]:
    layer.trainable = False

model = tf.keras.Sequential([
    base_model,
    tf.keras.layers.GlobalAveragePooling2D(),
    tf.keras.layers.Dense(256, activation='relu'),
    tf.keras.layers.Dropout(0.4),
    tf.keras.layers.Dense(2, activation='softmax')
])

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

print("Starting Pseudo-Label Training...")
history = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=10,
    class_weight=class_weights_dict
)

In [ ]:
import pandas as pd
import numpy as np
import os
import tensorflow as tf
from tensorflow.keras.preprocessing import image

# 1. เช็ก Mapping คลาส (ต้องรัน cell ที่มี train_generator ก่อนนะ)
label_map = train_generator.class_indices
label_map = dict((v, k) for k, v in label_map.items())

# 2. ตั้งค่าโฟลเดอร์ Test
sub_df = pd.read_csv('sample_submission.csv')
predictions = []

print(f"\nStarting Inference with TTA on {len(sub_df)} images...")
not_found_count = 0

for img_id in sub_df['id']:
    img_path = os.path.join(TEST_IMG_DIR, f"{img_id}.jpg")

    if os.path.exists(img_path):
        img = image.load_img(img_path, target_size=(384, 384))

        # ⚠️ เอาการหาร 255 ออก ให้เหลือแค่นี้:
        img_array = image.img_to_array(img)
        img_array = np.expand_dims(img_array, axis=0)

        # ทายผลจากภาพต้นฉบับ
        pred_original = model.predict(img_array, verbose=0)

        # พลิกภาพ (ซ้าย-ขวา) แล้วทายผล
        img_flipped = tf.image.flip_left_right(img_array)
        pred_flipped = model.predict(img_flipped, verbose=0)

        # เอาความมั่นใจมาเฉลี่ยกัน
        pred_final = (pred_original + pred_flipped) / 2.0

        # ดึงคลาสที่โมเดลมั่นใจที่สุดหลังจากเฉลี่ยแล้ว
        pred_idx = np.argmax(pred_final, axis=1)[0]
        pred_class = label_map[pred_idx]
        predictions.append(pred_class)
        # ==========================================

    else:
        not_found_count += 1
        predictions.append('0')

print(f"❌ หาไฟล์ไม่เจอ: {not_found_count} ภาพ")

# 3. บันทึกผลลัพธ์
sub_df['answer'] = predictions
sub_df.to_csv("submission_tta_4.csv", index=False)
print("✅ บันทึกไฟล์ 'submission_tta_4.csv' เรียบร้อยแล้ว ลุยส่งเลย!")